In [ ]:
import zipfile
import os

zip_path = "/content/daily+and+sports+activities.zip"
extract_path = "/content/DASD"

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("Dataset extracted successfully!")
print(os.listdir(extract_path))

Dataset extracted successfully!
['data']


In [ ]:
pip install pyts

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 57.0 MB/s eta 0:00:00


In [ ]:
from pyts.image import GramianAngularField
from sklearn.preprocessing import MinMaxScaler
import numpy as np

# load one sample
sample_file = "/content/DASD/data/a01/p1/s01.txt"

signal = np.loadtxt(sample_file, delimiter=",")

# normalize
scaler = MinMaxScaler(feature_range=(-1, 1))
normalized_sample = scaler.fit_transform(signal)

# GADF transformer
gadf = GramianAngularField(method='difference')

gadf_images = []

for ch in range(normalized_sample.shape[1]):

    channel_signal = normalized_sample[:, ch]

    channel_signal = channel_signal.reshape(1, -1)

    gadf_image = gadf.fit_transform(channel_signal)[0]

    gadf_images.append(gadf_image)

gadf_tensor = np.stack(
    gadf_images,
    axis=-1
)

print("GADF Tensor Shape:", gadf_tensor.shape)

GADF Tensor Shape: (125, 125, 45)


In [ ]:
from scipy.ndimage import zoom
import numpy as np

gadf_resized_channels = []

for ch in range(gadf_tensor.shape[-1]):

    img = gadf_tensor[:, :, ch]

    img_resized = zoom(
        img,
        (128/125, 128/125),
        order=1
    )

    gadf_resized_channels.append(img_resized)

gadf128_tensor = np.stack(
    gadf_resized_channels,
    axis=-1
)

print("Resized Shape:", gadf128_tensor.shape)

Resized Shape: (128, 128, 45)


In [ ]:
import os
import numpy as np
from pyts.image import GramianAngularField
from sklearn.preprocessing import MinMaxScaler
from scipy.ndimage import zoom

dataset_path = "/content/DASD/data"
save_path = "/content/GADF128_DATA"

os.makedirs(save_path, exist_ok=True)

gadf = GramianAngularField(method='difference')

sample_count = 0

activities = sorted(os.listdir(dataset_path))

for activity in activities:

    activity_path = os.path.join(dataset_path, activity)

    participants = sorted(os.listdir(activity_path))

    for participant in participants:

        participant_path = os.path.join(
            activity_path,
            participant
        )

        files = sorted(os.listdir(participant_path))

        for file in files:

            file_path = os.path.join(
                participant_path,
                file
            )

            signal_data = np.loadtxt(
                file_path,
                delimiter=','
            )

            scaler = MinMaxScaler(
                feature_range=(-1, 1)
            )

            normalized_signal = scaler.fit_transform(
                signal_data
            )

            gadf_channels = []

            for ch in range(
                normalized_signal.shape[1]
            ):

                signal = normalized_signal[:, ch]

                signal = signal.reshape(1, -1)

                gadf_image = gadf.fit_transform(
                    signal
                )[0]

                gadf_image = zoom(
                    gadf_image,
                    (128/125, 128/125),
                    order=1
                )

                gadf_channels.append(
                    gadf_image
                )

            gadf_tensor = np.stack(
                gadf_channels,
                axis=-1
            )

            save_name = (
                f"{activity}_{participant}_{file[:-4]}.npy"
            )

            np.save(
                os.path.join(
                    save_path,
                    save_name
                ),
                gadf_tensor.astype(
                    np.float32
                )
            )

            sample_count += 1

    print(f"Finished {activity}")

print("\nTotal Samples:", sample_count)

Finished a01
Finished a02
Finished a03
Finished a04
Finished a05
Finished a06
Finished a07
Finished a08
Finished a09
Finished a10
Finished a11
Finished a12
Finished a13
Finished a14
Finished a15
Finished a16
Finished a17
Finished a18
Finished a19

Total Samples: 9120
